# Training Horseflight’s neural evaluator

I kept the 781 → 256 → 32 → 1 network and the 5M/50K train-validation split fixed. I profiled the GPU loop first, swept the optimizer settings, then used the best runs to test mirror augmentation and different targets.

This experiment is separate from the weights bundled with the engine.

Use a Jupyter kernel with NumPy and PyTorch. The CPU checks need no data; training needs CUDA PyTorch and the original encoded caches under `HORSEFLIGHT_ROOT/research_data/`:

- `horseflight_n2_v1_1/encoded/`: `train_features.npy`, `train_cp.npy`, `val_features.npy`, `val_cp.npy`, and `n2_004_mirror_leak_mask.npy`
- `horseflight_n2_5m_v1/encoded/`: `extra_features.npy` and `extra_cp.npy`

The caches are derived from the [Lichess Open Database](https://database.lichess.org/) and are not included. Open from the repo root or `training/`. GPU jobs and checkpoint export are off until enabled.

## 1. Set up the runtime

Start with FP32 and no compilation. A different precision, compiler mode or dataset placement gets its own run directory so checkpoints aren't mixed.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

trainer_dir = Path.cwd()
if (trainer_dir / "training" / "horseflight_n2_006.py").is_file():
    trainer_dir /= "training"
SCRIPT_PATH = trainer_dir / "horseflight_n2_006.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError("Open Jupyter from the repository root or training/.")

sys.path.insert(0, str(trainer_dir))
import horseflight_n2_006 as hf

ROOT = Path(os.environ.get("HORSEFLIGHT_ROOT", trainer_dir.parent)).expanduser().resolve()
CHECKPOINTS = ROOT / "research_data/horseflight_n2_5m_v1/checkpoints"
GPUS = [f"cuda:{index}" for index in range(torch.cuda.device_count())]
PRECISION = "fp32"         # fp32, tf32, bf16
COMPILE_MODE = "none"      # none, default, reduce-overhead
DATASET_DEVICE = "cuda"    # cuda or cpu (placement, not the training device)
runtime_key = (PRECISION, COMPILE_MODE, DATASET_DEVICE)
run_name = "n2_006" if runtime_key == ("fp32", "none", "cuda") else "n2_006_" + "_".join(runtime_key)
OUTPUT_DIR = Path(os.environ.get("HORSEFLIGHT_OUTPUT", CHECKPOINTS / run_name)).expanduser().resolve()
print(f"PyTorch {torch.__version__} | GPUs: {GPUS or 'none'}")

## 2. Check the training contract on CPU

Check the sweep sizes, quantization-aware gradients, and that mirroring twice restores the features. The optional suite also checks checkpoint replay and saved promotions using synthetic data.

In [ ]:
a_configs = hf.campaign_a()
assert len(a_configs) == 162
assert len({hf.trial_id(config) for config in a_configs}) == 162
assert len(hf.campaign_b(a_configs[:3])) == 36

torch.manual_seed(12)
model = hf.HorseflightNNUE()
features = torch.full((2, 38), 781, dtype=torch.long)
features[0, :4] = torch.tensor([2, 18, 768, 769])
features[1, :3] = torch.tensor([65, 72, 771])
logits = model(features)
logits.square().sum().backward()
assert logits.shape == (2,) and torch.isfinite(logits).all()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
lut = hf.build_mirror_lut(torch.device("cpu"))
assert torch.equal(hf.fast_mirror(hf.fast_mirror(features, lut), lut), features)
print("Sweep, gradients and mirror checks passed.")

RUN_CPU_TESTS = False
if RUN_CPU_TESTS:
    subprocess.run([sys.executable, str(trainer_dir / "test_horseflight_n2_006.py")], check=True)

## 3. Measure throughput before the sweep

I timed unprofiled steps, then captured a separate trace to see where the time went. In the recorded FP32 run, profiling dropped the measured rate from about 681k to 29k examples/s. I used the trace to diagnose bottlenecks, not to report throughput.

Compare FP32, TF32, BF16 and BF16 with `reduce-overhead` on the same input. Use a fresh run directory when changing modes; do not resume an FP32 checkpoint in a different mode.

In [ ]:
def run_trainer(action, *flags):
    if not GPUS:
        raise RuntimeError("This step needs a CUDA device.")
    subprocess.run(
        [sys.executable, str(SCRIPT_PATH),
         "--action", action, "--root", str(ROOT), "--output", str(OUTPUT_DIR),
         "--devices", ",".join(GPUS), "--precision", PRECISION,
         "--compile", COMPILE_MODE, "--dataset-device", DATASET_DEVICE, *flags],
        check=True,
    )


RUN_BENCHMARK = False
RUN_PROFILER = False
if RUN_BENCHMARK:
    run_trainer("benchmark", "--steps", "60", "--warmup", "12")
if RUN_PROFILER:
    run_trainer("profile", "--steps", "20", "--warmup", "12")

## 4. Sweep optimizer settings

I chose the full 162 × 30-epoch sweep so every configuration had the same training budget. For a quicker pass, use synchronous halving: 162 → 40 → 10 trials at 3, 10 and 30 epochs. Promoted trials resume from checkpoints, with the scheduler set for 30 epochs from the start. Each GPU worker loads its copy of the dataset once.

In [ ]:
STRATEGY = "full"  # "halving" for the smaller sweep
RUN_CAMPAIGN_A = False
if RUN_CAMPAIGN_A:
    run_trainer("campaign", "--campaign", "a", "--strategy", STRATEGY)

## 5. Test symmetry and targets

Use the three best completed trials from the optimizer sweep to seed up to 36 variations. Partial runs don't qualify. This pass asks whether changing mirror augmentation, symmetry loss or target scaling helps once the base training settings are good.

In [ ]:
RUN_CAMPAIGN_B = False
if RUN_CAMPAIGN_B:
    run_trainer("campaign", "--campaign", "b", "--strategy", "full")

## 6. Check a second seed

Retrain the five best completed configurations from scratch on seed `20261529`. This is the check against picking a winner from a lucky initialization.

In [ ]:
RUN_CONFIRMATION = False
if RUN_CONFIRMATION:
    run_trainer("campaign", "--campaign", "confirmation", "--strategy", "full")

## 7. Compare on one metric

Rank on validation MSE against sigmoid(cp / 400), even when the training target differs. Keep partial trials out of the comparison; the held-out test set is not used to choose candidates.

The exported optimizer-sweep candidate scored **0.0083104**; the selected symmetry candidate scored **0.0082488**. These are validation scores from different configurations, not an isolated measure of symmetry loss or a playing-strength result.

In [ ]:
RUN_LEADERBOARD = False
if RUN_LEADERBOARD:
    rows = hf.export_leaderboard_from_disk(OUTPUT_DIR)
    completed = [row for row in rows if row["completed"]]
    print(f"{len(completed)} of {len(rows)} trials complete")
    for row in completed[:10]:
        print(f"{row['id']} | MSE {row['best_mse']:.7f} | "
              f"epoch {row['best_epoch']} | lr {row['lr']} | mirror {row['mirror_p']}")

## 8. Export the two candidates

Package the best checkpoints chosen for engine-side testing, with weights-only copies, run summaries and SHA-256 checksums. This step expects the original completed run directories.

In [ ]:
import hashlib
import json
import zipfile
from io import BytesIO

CANDIDATES = {
    "a_mse": CHECKPOINTS / "n2_006" / "f0d5d4752eacaa48",
    "b_mse_sym": CHECKPOINTS / "n2_006_b_fp32_compiled" / "88c4ea7ced0e7f72",
}


def export_checkpoint(name, source):
    best = source / "best.pt"
    checkpoint = torch.load(best, map_location="cpu", weights_only=True)
    config = json.loads((source / "config.json").read_bytes())
    summary = json.loads((source / "summary.json").read_bytes())
    if not (
        checkpoint["config"] == config
        and checkpoint["epoch"] == checkpoint["best_epoch"] == summary["best_epoch"]
        and summary["id"] == source.name
        and summary["completed"]
        and all(torch.isfinite(tensor).all().item() for tensor in checkpoint["model"].values())
    ):
        raise ValueError(f"Checkpoint and summary do not match a completed run: {source}")

    buffer = BytesIO()
    torch.save(checkpoint["model"], buffer)
    files = {file: (source / file).read_bytes() for file in ("best.pt", "config.json", "summary.json")}
    files.update({"weights_only.pt": buffer.getvalue(), SCRIPT_PATH.name: SCRIPT_PATH.read_bytes()})
    digests = {file: hashlib.sha256(data).hexdigest() for file, data in files.items()}
    manifest = {
        "candidate": name,
        "run_id": source.name,
        "best_epoch": checkpoint["best_epoch"],
        "validation_mse": summary["best_mse"],
        "dataset_id": checkpoint["dataset_id"],
        "sha256": digests,
    }

    out = CHECKPOINTS.parent / "handoffs"
    out.mkdir(parents=True, exist_ok=True)
    bundle = out / f"horseflight_{name}_{source.name}.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file, data in files.items():
            archive.writestr(file, data)
        archive.writestr("manifest.json", json.dumps(manifest, indent=2))

    with zipfile.ZipFile(bundle) as archive:
        for file, digest in digests.items():
            if hashlib.sha256(archive.read(file)).hexdigest() != digest:
                raise ValueError(f"Checksum mismatch: {file}")
    print(f"{bundle.name} | MSE {summary['best_mse']:.9f}")


RUN_EXPORT = False
if RUN_EXPORT:
    for name, source in CANDIDATES.items():
        export_checkpoint(name, source)